# FFA Non-Stationarity Diagnostic: Trend Testing and Outlier Screening

**Companion notebook to:** _Was That Flood an Outlier? A Python Diagnostic for Non-Stationarity in Flood Frequency Analysis_

**Data note — read this first:** every series in this notebook is **synthetically generated**, not an observed gauge record. The methods below are demonstrated on two constructed 55-year annual-maximum series — one built with no trend (a control), one with a known trend injected — so the notebook can show the diagnostics correctly telling the two apart. This validates the *method*; it says nothing about any specific real catchment. Run it on your own annual maximum series to get a result that means something for your own project.

## What this notebook does

1. Fits a stationary Log-Pearson III (LP3) distribution to an annual maximum series (`scipy.stats.pearson3` on log10-transformed data — the standard ARR 2019 / Bulletin 17C at-site approach).
2. Implements the Mann-Kendall trend test from scratch and validates it against a known-stationary and a known-trended series.
3. Builds a percentile/AEP diagnostic: fit LP3 to a historical record, then ask what annual exceedance probability the *stationary* model would assign to a new or recent event — the same question a sensitivity analysis excluding a large recent event is really asking.

**Key references:**
- Ball, J. et al. (2019), *Australian Rainfall and Runoff*, Book 3, Chapter 2 (LP3 at-site FFA).
- Mann, H.B. (1945), *Econometrica* 13(3): 245–259; Kendall, M.G. (1975), *Rank Correlation Methods*.
- Helsel, D.R. & Hirsch, R.M. (2002), *Statistical Methods in Water Resources*, USGS — the standard water-resources reference for the Mann-Kendall test and Sen's slope estimator.
- Brisbane City Council (2024), *Kedron Brook Flood Study, Vol. 1* — a real, published example of exactly this kind of sensitivity question applied to an actual record; see [the site's earlier commentary on it](https://lmillard79.github.io/insights/2025/03/15/kedron-brook-flood-study-2022-aep-analysis.html).

In [1]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

print('numpy:', np.__version__)
print('scipy:', __import__('scipy').__version__)

numpy: 2.4.6
scipy: 1.17.1


## 1. Synthetic annual maximum series

Two 55-year series, both drawn from a Log-Pearson III generator (so both are genuinely LP3-distributed by construction):

- **`series_stationary`** — no trend. A control.
- **`series_trended`** — a small linear drift (0.6%/year, compounding in log-space) added to the same generator.

55 years is a deliberately realistic record length for an Australian gauge — long by local standards, but still short relative to the return periods these methods are used to estimate.

In [2]:
rng_seed_base = 20260901
N_YEARS = 55

def make_series(trend_per_year=0.0, seed=0):
    """LP3-distributed annual maxima in log10 space, with optional linear
    drift added to the log-space location term to simulate a trend."""
    local_rng = np.random.default_rng(seed)
    skew, log_mean, log_sd = 0.3, 2.1, 0.22
    years = np.arange(N_YEARS)
    drift = trend_per_year * years
    log_q = stats.pearson3.rvs(skew=skew, loc=log_mean, scale=log_sd,
                                size=N_YEARS, random_state=local_rng)
    return 10 ** (log_q + drift)

series_stationary = make_series(trend_per_year=0.0, seed=1)
series_trended = make_series(trend_per_year=0.006, seed=2)
years = 1971 + np.arange(N_YEARS)  # arbitrary but realistic-looking calendar anchor

print(f'{N_YEARS} years, {years[0]}-{years[-1]}')
print(f'stationary — mean {series_stationary.mean():.1f}, median {np.median(series_stationary):.1f}')
print(f'trended    — mean {series_trended.mean():.1f}, median {np.median(series_trended):.1f}')

55 years, 1971-2025
stationary — mean 126.9, median 123.2
trended    — mean 223.4, median 172.3


## 2. Fit a stationary LP3 to each record

`scipy.stats.pearson3` parameterises directly as loc/scale/skew — fitting it to `log10(annual_max)` *is* Log-Pearson III. This uses MLE (`.fit()`); ARR 2019 practice often prefers L-moments for robustness with short records, but MLE is adequate here and keeps the dependency footprint to plain scipy.

In [3]:
def fit_lp3(annual_max):
    """MLE fit of LP3 (Pearson III on log10-transformed data). Returns skew, loc, scale."""
    log_q = np.log10(annual_max)
    skew, loc, scale = stats.pearson3.fit(log_q)
    return skew, loc, scale

def lp3_quantile(aep, skew, loc, scale):
    """Flow value at a given AEP (e.g. 0.01 for the 1% AEP / 100-year event)."""
    return 10 ** stats.pearson3.ppf(1 - aep, skew=skew, loc=loc, scale=scale)

def lp3_aep(value, skew, loc, scale):
    """AEP implied by a fitted LP3 for a given flow value."""
    return 1 - stats.pearson3.cdf(np.log10(value), skew=skew, loc=loc, scale=scale)

for name, series in [('stationary', series_stationary), ('trended', series_trended)]:
    skew, loc, scale = fit_lp3(series)
    q1pct = lp3_quantile(0.01, skew, loc, scale)
    print(f'{name:11s} skew={skew:+.3f} loc={loc:.3f} scale={scale:.3f}  '
          f'-> fitted 1% AEP (100-yr) flow = {q1pct:.1f}')

stationary  skew=-0.116 loc=2.065 scale=0.183  -> fitted 1% AEP (100-yr) flow = 299.3
trended     skew=+0.577 loc=2.280 scale=0.226  -> fitted 1% AEP (100-yr) flow = 792.3


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

for ax, (name, series) in zip(axes, [('Stationary (control)', series_stationary),
                                       ('Trended (+0.6%/yr log-space drift)', series_trended)]):
    skew, loc, scale = fit_lp3(series)
    ax.scatter(years, series, s=22, color='steelblue', zorder=3, label='Annual maximum')
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Year')
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Annual maximum flow (m³/s, synthetic units)')
axes[0].legend(fontsize=9)
fig.suptitle('Two synthetic 55-year annual-maximum series', fontsize=12)
plt.tight_layout()
plt.savefig('../../images/2026-09_ffa-nonstationarity-series.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size ... with Axes>

## 3. Mann-Kendall trend test, from scratch

A non-parametric test for monotonic trend: for every pair of observations, count whether the later one is larger (+1), smaller (-1), or tied (0), sum the signs into a statistic *S*, and compare *S* against its expected variance under the null hypothesis of no trend. Sen's slope is the median of all pairwise slopes — a robust trend-magnitude estimate that isn't pulled around by one or two outliers the way ordinary least squares would be.

This is the standard test used in the water-resources trend literature (Helsel & Hirsch) precisely because it doesn't assume a particular distribution for the data — appropriate for a right-skewed annual maximum series.

**The point of running it on both series below is to validate the implementation itself**: it should correctly fail to reject the null on the stationary control and correctly reject it on the trended series. If it didn't, the test would not be trustworthy to run on anything real.

In [5]:
def mann_kendall(x):
    """Mann-Kendall trend test. Returns S, Z, two-sided p-value, and Sen's slope."""
    n = len(x)
    s = 0
    for k in range(n - 1):
        s += np.sum(np.sign(x[k+1:] - x[k]))
    s = int(s)
    var_s = n * (n - 1) * (2 * n + 5) / 18  # no tie correction needed: continuous synthetic data
    if s > 0:
        z = (s - 1) / np.sqrt(var_s)
    elif s < 0:
        z = (s + 1) / np.sqrt(var_s)
    else:
        z = 0.0
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    slopes = [(x[j] - x[i]) / (j - i) for i in range(n - 1) for j in range(i + 1, n)]
    sen_slope = float(np.median(slopes))
    return s, z, p, sen_slope

for name, series in [('Stationary (control)', series_stationary),
                      ('Trended', series_trended)]:
    s, z, p, sen = mann_kendall(series)
    verdict = 'SIGNIFICANT trend (p < 0.05)' if p < 0.05 else 'no significant trend (p >= 0.05)'
    print(f'{name:22s} S={s:5d}  Z={z:+.2f}  p={p:.4f}  Sen\'s slope={sen:+.3f}/yr  -> {verdict}')

assert mann_kendall(series_stationary)[2] >= 0.05, 'control series should NOT show a significant trend'
assert mann_kendall(series_trended)[2] < 0.05, 'trended series SHOULD show a significant trend'
print()
print('Self-check passed: the test correctly distinguishes the two series.')

Stationary (control)   S=   41  Z=+0.29  p=0.7715  Sen's slope=+0.110/yr  -> no significant trend (p >= 0.05)
Trended                S=  319  Z=+2.31  p=0.0210  Sen's slope=+1.459/yr  -> SIGNIFICANT trend (p < 0.05)

Self-check passed: the test correctly distinguishes the two series.


## 4. The percentile/AEP diagnostic

This is the question a "how unusual was that event, really" discussion is actually asking, made explicit: fit LP3 to a historical record, then read off what AEP the *stationary* model — the one that assumes nothing has changed — assigns to a given value.

Two constructed scenarios below (again: constructed, not observed) show the two ends of what the answer can look like:

- **Scenario A** lands around the 7% AEP mark under the historical-only fit — comparable in AEP terms to what the [Kedron Brook Flood Study](https://lmillard79.github.io/insights/2025/03/15/kedron-brook-flood-study-2022-aep-analysis.html) actually found for the real 2022 Brisbane event: rare, but well within what a stationary model would expect to produce occasionally.
- **Scenario B** lands under 0.5% AEP under the same historical-only fit — the kind of result that's a legitimate trigger to look closer, not proof of anything on its own.

In [6]:
history = series_stationary  # fit to the full 55-yr historical-only record
skew, loc, scale = fit_lp3(history)
print(f'Historical-only LP3 fit: skew={skew:+.3f} loc={loc:.3f} scale={scale:.3f}')
print()

target_aeps = {
    'Scenario A -- rare, not extreme': 0.07,
    'Scenario B -- statistically inconsistent': 0.004,
}
scenario_values = {}
for label, target_aep in target_aeps.items():
    value = lp3_quantile(target_aep, skew, loc, scale)
    recovered_aep = lp3_aep(value, skew, loc, scale)  # round-trip check
    pct_of_record_exceeded = float(np.mean(history < value)) * 100
    scenario_values[label] = value
    print(f'{label}')
    print(f'    constructed value = {value:.1f}')
    print(f'    stationary-model AEP = {recovered_aep*100:.2f}%  (~1-in-{1/recovered_aep:.0f} yr)')
    print(f'    exceeds {pct_of_record_exceeded:.0f}% of the 55-yr historical record')
    print()

Historical-only LP3 fit: skew=-0.116 loc=2.065 scale=0.183

Scenario A -- rare, not extreme
    constructed value = 214.6
    stationary-model AEP = 7.00%  (~1-in-14 yr)
    exceeds 96% of the 55-yr historical record

Scenario B -- statistically inconsistent
    constructed value = 338.9
    stationary-model AEP = 0.40%  (~1-in-250 yr)
    exceeds 98% of the 55-yr historical record



In [7]:
aep_range = np.logspace(np.log10(0.001), np.log10(0.99), 300)
flow_curve = [lp3_quantile(a, skew, loc, scale) for a in aep_range]

# Cunnane plotting positions for the historical record
sorted_hist = np.sort(history)[::-1]
n = len(sorted_hist)
ranks = np.arange(1, n + 1)
cunnane_aep = (ranks - 0.4) / (n + 0.2)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(aep_range * 100, flow_curve, color='steelblue', lw=2, label='Stationary LP3 fit (historical-only)')
ax.scatter(cunnane_aep * 100, sorted_hist, color='dimgray', s=25, zorder=3,
           label='55-yr historical record (Cunnane plotting position)')

markers = {'Scenario A -- rare, not extreme': ('o', 'seagreen'),
           'Scenario B -- statistically inconsistent': ('^', 'firebrick')}
for label, value in scenario_values.items():
    aep = lp3_aep(value, skew, loc, scale)
    marker, color = markers[label]
    ax.scatter([aep * 100], [value], color=color, s=160, marker=marker, zorder=5,
               edgecolor='black', linewidth=0.8, label=label)

ax.set_xscale('log')
ax.invert_xaxis()
ax.set_xlabel('Annual Exceedance Probability, %  (log scale)')
ax.set_ylabel('Flow (m³/s, synthetic units)')
ax.set_title('Where a new event falls against a stationary historical-only fit', fontsize=12)
ax.grid(True, which='both', alpha=0.3)
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.savefig('../../images/2026-09_ffa-nonstationarity-diagnostic.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size ... with Axes>

## 5. What this diagnostic can and can't tell you

**What it can do:**
- Give a precise, repeatable answer to "how rare would this be if nothing had changed?" — replacing gut-feel language like "unprecedented" with an actual AEP against a stated model.
- Flag when a record is being dominated by one or two large events (Scenario B territory), which is a legitimate reason to run a sensitivity analysis excluding them, exactly as the Kedron Brook Flood Study did.
- Give the Mann-Kendall test a like-for-like check on whether the annual maximum series itself shows a monotonic trend, independent of any one event.

**What it can't do:**
- Tell you *why* an event fell where it did. A Scenario-B-type result is consistent with several explanations — genuine climate-driven non-stationarity, a short and unrepresentative record, measurement or rating-curve error at extreme flows, or simply an unlucky draw from a correctly-specified stationary distribution (rare events do happen under stationarity — that's what "rare" means). Telling those apart is a different, harder question than the one this diagnostic answers.
- Substitute for formal climate attribution. Attribution studies (e.g. the World Weather Attribution methodology) use climate model ensembles to estimate how much more likely or intense a specific event was made by anthropogenic warming — a materially different question, answered with different tools, by people whose primary discipline is climate science. A flood engineer running a Mann-Kendall test is not doing that, and shouldn't present it as though it were.
- Replace professional judgement on record length and representativeness. 55 years is generous by Australian standards; many real gauge records are much shorter, and both the LP3 fit and the trend test above should be read with correspondingly wider uncertainty the shorter the record actually is.

## References

- Ball, J. et al. (2019). *Australian Rainfall and Runoff.* Book 3, Chapter 2.
- Mann, H.B. (1945). Nonparametric tests against trend. *Econometrica* 13(3): 245–259.
- Kendall, M.G. (1975). *Rank Correlation Methods.* Griffin.
- Helsel, D.R. & Hirsch, R.M. (2002). *Statistical Methods in Water Resources.* USGS Techniques of Water-Resources Investigations, Book 4, Chapter A3.
- Brisbane City Council (2024). *Kedron Brook Flood Study, Vol. 1* (for information only, not Council policy).
- World Weather Attribution — [worldweatherattribution.org](https://www.worldweatherattribution.org/) — for what formal event attribution actually involves.